# Lab 9.2 &mdash; Probes That Can Actually Fail

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Write <code>/healthz</code> and <code>/readyz</code> as real FastAPI routes that answer different questions
- Decide which failures restart a container and which only drain traffic
- Simulate the kubelet and price <code>failureThreshold</code> in seconds of traffic
- Find out what a readiness check that calls the model costs per day

> **How this lab works.** You write real FastAPI, Pydantic, LangChain and Kubernetes-manifest
> code. Fill every `BLANK`, then run the **Self-check** cell under each section &mdash; those
> assert on the *objects you built* (a route table, a request contract, a compiled tool, a
> manifest dict), so they are deterministic. **No graded cell needs a cluster, a running server
> or a model.** Cells marked **Run it for real** put your code in front of the sandbox model,
> your own namespace or the tracing backend; if any of those is unreachable they print how to
> fix it instead of crashing. The score line is feedback, not a grade.

> **Nothing here needs a cluster.** The kubelet's loop is twenty lines, and you can
> run a thirty-second gateway outage through it in a millisecond. That is a better
> way to learn what `failureThreshold` means than waiting for one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable, Optional

WORK = os.path.join("/tmp", "awmas-lab-9-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because a deployment lab makes a lot of small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=LLM_API_KEY,
                          temperature=temperature, extra_body=NO_THINK)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")
print("namespace:", APP_NS or "(unknown -- no graded cell needs it)")

## Concept

Kubernetes asks a pod two different questions and most services answer both the same way.

- **Liveness** &mdash; *is this process broken beyond recovery?* A failure here **restarts the
  container**. It must not depend on anything you do not control.
- **Readiness** &mdash; *should this replica receive traffic right now?* A failure here **removes
  the pod from the Service** and nothing else. It may depend on everything.

An agent service makes the distinction sharp, because its main dependency &mdash; the model
gateway &mdash; is remote, shared, and occasionally slow.

## Section 1 &mdash; Two endpoints, two questions

Both probes are real FastAPI routes returning real `JSONResponse` objects, so the self-checks
below read the same `status_code` the kubelet would. The trap in this section is that a probe
reads the **status code** and never looks at the body.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse

# What this replica knows about ITSELF. A readiness check reads these; it does not go
# and find out, because Section 3 prices what "going and finding out" costs.
STATE = {"agent_built": True, "in_flight": 0, "gateway_failing": False}
MAX_IN_FLIGHT = 32

probes = FastAPI(title="agent-probes")


def readiness_reason() -> Optional[str]:
    """None means ready. A string means drain me, with a reason a human can read.

    Every branch is local and costs microseconds. `gateway_failing` is a flag the REQUEST
    PATH sets when it sees the gateway fail -- readiness reads it, it never generates a
    call of its own.
    """
    if not STATE["agent_built"]:
        return "agent not constructed yet"
    if STATE["in_flight"] >= MAX_IN_FLIGHT:
        return "at capacity"
    if STATE["gateway_failing"]:
        return "gateway failing on the request path"
    return None


@probes.get("/healthz")
def healthz() -> JSONResponse:
    """Liveness. Is THIS PROCESS broken beyond recovery? Checks nothing remote, on purpose."""
    return JSONResponse(status_code=200, content={"status": "ok"})


@probes.get("/readyz")
def readyz() -> JSONResponse:
    """Readiness. Should this replica be sent a request right now?"""
    reason = readiness_reason()
    if reason is None:
        return JSONResponse(status_code=200, content={"ready": True})
    # TODO: the kubelet reads the STATUS CODE and never looks at the body. Return the
    # code that means "not right now -- take me out of the Service".
    return JSONResponse(status_code=BLANK, content={"ready": False, "why": reason})


@probes.get("/readyz-decorative")
def readyz_that_cannot_fail() -> JSONResponse:
    """The bug this lab exists for. Looks careful. Is decorative."""
    return JSONResponse(status_code=200, content={"ready": readiness_reason() is None})

print("routes:", sorted(r.path for r in probes.routes if hasattr(r, "methods")))

In [ ]:
def probe_for(condition: str) -> str:
    """Which probe owns this condition?

    "liveness"  -- failing it RESTARTS the container
    "readiness" -- failing it removes the pod from the Service, and nothing else
    """
    table = {
        "the event loop is deadlocked and serves nothing": "liveness",
        "the config file failed to parse at start-up":     "liveness",

        # TODO: a restart cannot bring a remote gateway back, and three replicas
        # restarting through a thirty-second blip turns it into a ten-minute outage.
        "the model gateway is returning 503":              BLANK,

        # TODO: this replica already has thirty-two requests in flight. It is not
        # broken, and another replica can take the next one.
        "this replica is at its in-flight limit":          BLANK,
    }
    return table[condition]

In [ ]:
# --- Self-check: Section 1   (route objects and status codes -- no cluster, no model)
def routes() -> dict:
    return {r.path: set(r.methods) for r in probes.routes if hasattr(r, "methods")}

def with_state(fn, **overrides):
    """Call fn() with STATE temporarily overridden, then put it back."""
    was = dict(STATE)
    STATE.update(overrides)
    try:
        return fn()
    finally:
        STATE.clear()
        STATE.update(was)

check("the app serves both probes on GET",
      lambda: "GET" in routes()["/healthz"] and "GET" in routes()["/readyz"])
check("readiness passes while everything is fine",
      lambda: readyz().status_code == 200)
check("READINESS FAILS WITH A STATUS CODE when the gateway is failing",
      lambda: with_state(lambda: readyz().status_code, gateway_failing=True) == 503,
      "503 is what removes the pod from the Service; the body is never read")
check("...and says why, for the human reading `kubectl describe pod`",
      lambda: with_state(lambda: json.loads(readyz().body)["why"], gateway_failing=True))
check("readiness also fails when this replica is full",
      lambda: with_state(lambda: readyz().status_code, in_flight=MAX_IN_FLIGHT) == 503)
check("liveness passes anyway, because the process is fine",
      lambda: with_state(lambda: healthz().status_code, gateway_failing=True) == 200,
      "restarting it would not bring the gateway back")
check("the decorative version says the right thing in the body",
      lambda: with_state(lambda: json.loads(readyz_that_cannot_fail().body)["ready"],
                         gateway_failing=True) is False)
check("...and STILL RETURNS 200, so that probe can never fail",
      lambda: with_state(lambda: readyz_that_cannot_fail().status_code,
                         gateway_failing=True) == 200,
      "a readiness check that removes the pod from the Service exactly never")
check("a failing gateway drains traffic; it does not restart the container",
      lambda: probe_for("the model gateway is returning 503") == "readiness",
      "a restart cannot fix somebody else's service, and three of them make it worse")
check("a full replica drains too",
      lambda: probe_for("this replica is at its in-flight limit") == "readiness")
check("a deadlocked process is the one thing a restart does fix",
      lambda: probe_for("the event loop is deadlocked and serves nothing") == "liveness")

## Section 2 &mdash; The kubelet's loop

`periodSeconds`, `failureThreshold` and `initialDelaySeconds` are the whole of it. Writing the
loop once tells you what the numbers cost, in seconds of traffic sent to a replica that cannot
serve it. The loop is given; the configuration is yours.

In [ ]:
PERIOD            = 5      # periodSeconds
FAILURE_THRESHOLD = 3      # failureThreshold
INITIAL_DELAY     = 30     # initialDelaySeconds -- must exceed real start-up time
COLD_START        = 20     # how long this app takes to be able to answer at all
REPLICAS          = 3
GATEWAY_DOWN      = (30, 60)     # the gateway is unreachable for 30 seconds
HORIZON           = 120


def gateway_up(t: int) -> bool:
    return not (GATEWAY_DOWN[0] <= t < GATEWAY_DOWN[1])


def probe_result(endpoint: str, t: int, replica: dict) -> int:
    """What `endpoint` returns for this replica at second t."""
    if t - replica["started"] < COLD_START:
        return 503                                   # not listening yet
    if endpoint == "healthz":
        return 200                                   # the process is up; it checks nothing else
    return 200 if gateway_up(t) else 503             # readyz consults the gateway flag


def act_now(consecutive_failures: int) -> bool:
    """One bad probe is a blip. The kubelet acts on failureThreshold in a row."""
    return consecutive_failures >= FAILURE_THRESHOLD

In [ ]:
def good_config() -> dict:
    """The configuration that drains traffic without destroying warm processes."""
    return {
        # TODO: liveness must not depend on anything a restart cannot fix. Which of the
        # two endpoints belongs here -- "healthz" or "readyz"?
        "liveness":  BLANK,
        "readiness": "readyz",
    }


def bad_config() -> dict:
    """Both probes pointed at the same endpoint. The most common mistake there is."""
    return {"liveness": "readyz", "readiness": "readyz"}

In [ ]:
def simulate(liveness: str, readiness: str, horizon: int = HORIZON) -> dict:
    """Run REPLICAS replicas through the outage under one probe configuration.

    Returns restarts, the seconds with no ready replica, and the seconds spent serving
    traffic from a replica that cannot actually answer.
    """
    reps = [{"started": -INITIAL_DELAY - 10, "live": 0, "ready_f": 0, "ready": True,
             "restarts": 0} for _ in range(REPLICAS)]
    served = {}
    for t in range(horizon):
        for r in reps:
            if t - r["started"] < INITIAL_DELAY:      # initialDelaySeconds: no probing yet
                r["ready"] = False
                continue
            if t % PERIOD:
                continue
            if probe_result(liveness, t, r) != 200:
                r["live"] += 1
                if act_now(r["live"]):                # liveness failing RESTARTS the container
                    r.update(started=t, live=0, ready_f=0, ready=False,
                             restarts=r["restarts"] + 1)
                    continue
            else:
                r["live"] = 0
            if probe_result(readiness, t, r) != 200:
                r["ready_f"] += 1
                if act_now(r["ready_f"]):             # readiness failing only DRAINS traffic
                    r["ready"] = False
            else:
                r["ready_f"], r["ready"] = 0, True
        served[t] = sum(1 for r in reps if r["ready"])

    down = [t for t in range(horizon) if served[t] == 0]
    broken = [t for t in range(horizon) if served[t] > 0 and not gateway_up(t)]
    return {"restarts": sum(r["restarts"] for r in reps),
            "blackout_s": len(down),
            "recovered_at": (max(down) + 1) if down else None,
            "serving_while_broken_s": len(broken)}

In [ ]:
# --- Self-check: Section 2   (pure simulation -- no cluster)
def good():
    return simulate(**good_config())

def bad():
    return simulate(**bad_config())

check("one failed probe is not enough to act on",
      lambda: act_now(1) is False)
check("three in a row is",
      lambda: act_now(FAILURE_THRESHOLD) is True)
check("so the kubelet waits period x threshold = 15s before it does anything",
      lambda: PERIOD * FAILURE_THRESHOLD == 15,
      "that is 15 seconds of traffic to a replica that is already failing")
check("liveness on healthz survives the outage with NO restarts",
      lambda: good()["restarts"] == 0,
      "the process was never broken -- somebody else's gateway was")
check("POINTING LIVENESS AT THE DEPENDENCY RESTARTS EVERY REPLICA",
      lambda: bad()["restarts"] >= REPLICAS,
      "a 30-second gateway blip becomes a fleet-wide restart")
check("...and the restarts cause a blackout the good config never has",
      lambda: bad()["blackout_s"] > good()["blackout_s"])
check("...that outlasts the outage itself, because cold start is 20s",
      lambda: bad()["recovered_at"] > GATEWAY_DOWN[1])
check("the good config still drains traffic during the outage",
      lambda: good()["serving_while_broken_s"] < (GATEWAY_DOWN[1] - GATEWAY_DOWN[0]),
      "readiness did its job: the pods left the Service without being killed")

def _compare():
    for label, cfg in (("liveness=healthz (good)", good_config()),
                       ("liveness=readyz  (bad) ", bad_config())):
        r = simulate(**cfg)
        print(f"  {label}  restarts={r['restarts']:>2}  blackout={r['blackout_s']:>3}s  "
              f"recovered_at={r['recovered_at']}")
guard(_compare)

### Read it

The bad configuration is not exotic. It is what you get by writing the readiness endpoint first,
liking it, and pointing both probes at it &mdash; which reads as *thorough*.

What it actually does is convert a dependency's thirty-second blip into a fleet-wide restart,
and then add your own cold-start time on top. The service is down for longer than the thing it
depends on was.

Readiness alone would have removed the pods from the Service and put them back the moment the
gateway returned, with no process killed and no cache lost.

## Section 3 &mdash; What a readiness check costs

Readiness runs on every replica, forever. That makes it the only code in your service whose
cost is set by `periodSeconds` rather than by traffic.

In [ ]:
def probe_load(replicas: int, period_s: float, check_seconds: float) -> dict:
    """What a dependency-checking readiness probe costs per minute, across the fleet."""
    per_replica_per_min = 60 / period_s
    calls = replicas * per_replica_per_min
    return {"calls_per_min": calls,
            "gateway_seconds_per_min": calls * check_seconds,
            "calls_per_day": calls * 60 * 24}

In [ ]:
# --- Self-check: Section 3
CHEAP = probe_load(REPLICAS, PERIOD, 0.001)     # reads a local flag, like readiness_reason()
REAL  = probe_load(REPLICAS, PERIOD, 0.800)     # calls the model for one token

check("a cheap readiness check costs nothing measurable",
      lambda: CHEAP["gateway_seconds_per_min"] < 0.1)
check("the same probe that calls the model does not",
      lambda: REAL["gateway_seconds_per_min"] > 25)
check("and it does it 51,840 times a day at three replicas",
      lambda: REAL["calls_per_day"] == 51840)
check("halving periodSeconds doubles all of it",
      lambda: probe_load(REPLICAS, PERIOD / 2, 0.8)["calls_per_day"]
              == REAL["calls_per_day"] * 2)
check("readiness_reason() is on the cheap side of that line",
      lambda: readiness_reason() is None or isinstance(readiness_reason(), str),
      "every branch in it reads STATE -- no network, no model, no clock skew")

def _cost():
    print(f"  cheap check : {CHEAP['gateway_seconds_per_min']:.3f}s of gateway time per minute")
    print(f"  model check : {REAL['gateway_seconds_per_min']:.1f}s per minute, "
          f"{REAL['calls_per_day']:,.0f} calls per day")
    print("  A readiness probe that calls the model is a load generator you did not plan for,")
    print("  pointed at the dependency you are worried about.")
guard(_cost)

### So what should readiness check?

Exactly what `readiness_reason()` checks: things that are **local and cheap**. Is the client
constructed, is the config loaded, is the in-flight count below the limit, and &mdash; for the
gateway &mdash; a **flag that the request path sets** when it sees failures, rather than a call
the probe generates itself.

That pattern also removes the failure mode where a struggling gateway gets an extra 52,000
calls a day from the health checks of the very service that is waiting on it.

## Run it for real

Time a minimal call to the sandbox gateway, then price the probe you would have written if
`readyz` had called the model.

In [ ]:
if llm_ready():
    def _price_it():
        t0 = time.perf_counter()
        reply = ask("ok")
        latency = time.perf_counter() - t0
        if reply.startswith("<model unavailable"):
            print(reply)                      # say so, rather than timing a failure
            return
        load = probe_load(REPLICAS, PERIOD, latency)
        print(f"  one minimal call        : {latency:.2f}s")
        print(f"  as a readiness probe    : {load['gateway_seconds_per_min']:.1f}s of gateway "
              f"time per minute, {load['calls_per_day']:,.0f} calls/day")
        print(f"  at 30 participants      : {load['calls_per_day'] * 30:,.0f} calls/day "
              f"before anyone asks a question")
    guard(_price_it)

In [ ]:
score()

## Your turn

1. Add a `startupProbe` to the simulation and remove `initialDelaySeconds`. Show the case it
   handles better: an app whose start-up time varies between 5 and 90 seconds.
2. Wire `STATE["gateway_failing"]` to something real: set it in `ask_endpoint`'s error path
   from Lab 9.1, with a cool-down. Then find its failure mode &mdash; what happens when there
   is no traffic at all?
3. `terminationGracePeriodSeconds` is the other half of draining. Work out what an agent request
   that has been running for 90 seconds should do when the pod is told to stop.